# Inventory Management Optimization

The purpose of this project is to **optimize inventory management** for a medium-sized manufacturing company by analyzing key aspects of inventory data. The goals include:

1. **Improving Data Quality**: Ensure the inventory data is clean, consistent, and ready for analysis.

2. **Understanding Lead Times**: Analyze supplier performance by calculating and visualizing lead times.

3. **Preventing Stockouts**: Calculate safety stock levels to avoid stockouts and ensure product availability.

4. **Reducing Excess Inventory**: Identify items with excess inventory and recommend strategies to reduce carrying costs.

5. **Evaluating Vendor Performance**: Analyze vendor performance based on total spend and payment cycles.

6. **Identifying Top Performers**: Determine the top-selling items and stores to focus on high-revenue products and locations.

7. **Optimizing Order Quantities**: Calculate the Economic Order Quantity (EOQ) to minimize ordering and carrying costs.

This project seeks to find areas where the company can reduce costs, improve efficiency, and enhance customer satisfaction.

###Data 

The project uses the following CSV files from the Inventory_Analysis_Case_Study.zip file:

    InvoicePurchases12312016.csv

    EndInvFINAL12312016.csv

    BegInvFINAL12312016.csv

    2017PurchasePricesDec.csv

    SalesFINAL12312016.csv

    PurchasesFINAL12312016.csv

## 1. Import necessary libraries

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import zipfile
import logging
from datetime import datetime

Load the inventory data from a zip file into pandas DataFrames for analysis. The code reads the CSV files from the zip file and stores them in a dictionary of DataFrames.

In [2]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set pandas display options
pd.set_option('display.max_columns', 200)
plt.style.use('ggplot')

In [3]:
# Load data from the zip file
# Function to load data from the zip file
def load_data_from_zip(zip_path):
    """
    Load CSV files from a zip archive into pandas DataFrames.
    
    Args:
        zip_path (str): Path to the zip file.
    
    Returns:
        dict: A dictionary of DataFrames with keys as file names.
    """
    dataframes = {}
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # List of files to extract
            files_to_extract = [
                'InvoicePurchases12312016.csv',
                'EndInvFINAL12312016.csv',
                'BegInvFINAL12312016.csv',
                '2017PurchasePricesDec.csv',
                'SalesFINAL12312016.csv',
                'PurchasesFINAL12312016.csv'
            ]
            
            # Load each file into a DataFrame
            for file in files_to_extract:
                with zip_ref.open(file) as f:
                    df_name = file.replace('.csv', '')
                    dataframes[df_name] = pd.read_csv(f)
                    logging.info(f"Successfully loaded {file} as {df_name}")
    
    except Exception as e:
        logging.error(f"An error occurred while loading data: {e}")
    
    return dataframes

## 2. Clean Data

In [4]:
# Function to clean data
def clean_data(df):
    """
    Clean the DataFrame by handling missing values, duplicates, and whitespace.
    
    Args:
        df (pd.DataFrame): The DataFrame to clean.
    
    Returns:
        pd.DataFrame: The cleaned DataFrame.
    """
    # Handle missing values
    if df.isna().sum().sum() > 0:
        logging.info("Handling missing values...")
        df.dropna(inplace=True)
    
    # Remove duplicates
    if df.duplicated().sum() > 0:
        logging.info("Removing duplicates...")
        df.drop_duplicates(inplace=True)
    
    # Remove leading/trailing whitespace in object columns
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()
    
    return df

In [5]:
# Function to preprocess data
def preprocess_data(df, date_columns=None):
    """
    Preprocess the DataFrame by typecasting dates and handling irregularities.
    
    Args:
        df (pd.DataFrame): The DataFrame to preprocess.
        date_columns (list): List of columns to typecast as datetime.
    
    Returns:
        pd.DataFrame: The preprocessed DataFrame.
    """
    if date_columns:
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            logging.info(f"Typecasted {col} to datetime.")
    
    return df

## 3. Analyzing Lead Times

Understand supplier performance by calculating and visualizing lead times. The code calculates the lead time (time between placing an order and receiving it) for each product and summarizes the max, min, average, and standard deviation of lead times.

In [6]:
# Function to analyze lead time
def analyze_lead_time(purchases_df):
    """
    Analyze lead time for purchases.
    
    Args:
        purchases_df (pd.DataFrame): The purchases DataFrame.
    
    Returns:
        pd.DataFrame: A summary of lead time statistics.
    """
    purchases_df['LeadTime'] = (purchases_df['ReceivingDate'] - purchases_df['PODate']).dt.days
    
    leadtime_summary = purchases_df.groupby(['Brand', 'Description']).agg(
        Max_leadtime=('LeadTime', 'max'),
        Min_leadtime=('LeadTime', 'min'),
        Avg_leadtime=('LeadTime', 'mean'),
        Std_leadtime=('LeadTime', 'std')
    ).round(3).reset_index()
    
    return leadtime_summary

## 4. Calculating Safety Stock

Determine the optimal safety stock level to avoid stockouts. The code calculates the average daily sales and uses it, along with lead time variability, to determine the safety stock level for each product.

In [7]:
# Function to calculate safety stock
def calculate_safety_stock(sales_df, leadtime_summary):
    """
    Calculate safety stock for each product.
    
    Args:
        sales_df (pd.DataFrame): The sales DataFrame.
        leadtime_summary (pd.DataFrame): The lead time summary DataFrame.
    
    Returns:
        pd.DataFrame: A DataFrame with safety stock levels.
    """
    # Calculate average daily sales
    sales_days_range = (sales_df['SalesDate'].max() - sales_df['SalesDate'].min()).days
    sales_summary = sales_df.groupby(['Brand', 'Description']).agg(
        Total_sales=('SalesQuantity', 'sum'),
        Std_sales=('SalesQuantity', 'std')
    ).reset_index()
    sales_summary['avg_daily_sales'] = (sales_summary['Total_sales'] / sales_days_range).round(3)
    
    # Merge sales summary with lead time summary
    safety_stock_data = pd.merge(sales_summary, leadtime_summary, on=['Brand', 'Description'])
    
    # Calculate safety stock
    safety_stock_data['safety_stock'] = np.ceil(
        (safety_stock_data['avg_daily_sales'] + 1.5 * safety_stock_data['Std_sales']) *
        (safety_stock_data['Avg_leadtime'] + 1.5 * safety_stock_data['Std_leadtime'])
    )
    
    return safety_stock_data

## 5. Identifying Excess Inventory

Identify items with excess inventory to reduce carrying costs. The code compares the current inventory levels with the calculated safety stock to identify items with excess inventory.

In [8]:
# Function to identify excess inventory
def identify_excess_inventory(safety_stock_data, inventory_df):
    """
    Identify excess inventory by comparing safety stock with current inventory.
    
    Args:
        safety_stock_data (pd.DataFrame): The safety stock DataFrame.
        inventory_df (pd.DataFrame): The inventory DataFrame.
    
    Returns:
        pd.DataFrame: A DataFrame with excess inventory details.
    """
    # Summarize current inventory
    inventory_summary = inventory_df.groupby(['Brand', 'Description']).agg(
        Onhand_stock=('onHand', 'sum')
    ).reset_index()
    
    # Merge safety stock with current inventory
    inventory_status = pd.merge(safety_stock_data, inventory_summary, on=['Brand', 'Description'])
    
    # Calculate excess inventory
    inventory_status['excess_inventory'] = inventory_status['Onhand_stock'] - inventory_status['safety_stock']
    inventory_status['status'] = np.where(
        inventory_status['excess_inventory'] > 0,
        'Excess',
        np.where(inventory_status['excess_inventory'] == 0, 'Balanced', 'Shortage'))
    
    return inventory_status

## 6. Analyzing Vendor Performance

Evaluate vendor performance based on total spend and payment cycles. The code calculates the total spend, number of orders, and average payment cycle for each vendor.

In [9]:
# Function to analyze vendor performance
def analyze_vendor_performance(purchases_df):
    """
    Analyze vendor performance and payment cycles.
    
    Args:
        purchases_df (pd.DataFrame): The purchases DataFrame.
    
    Returns:
        pd.DataFrame: A summary of vendor performance.
    """
    # Calculate payment cycle
    purchases_df['payment_cycle'] = (purchases_df['PayDate'] - purchases_df['InvoiceDate']).dt.days
    
    # Summarize vendor performance
    vendor_summary = purchases_df.groupby('VendorName').agg(
        Total_orders=('PONumber', 'count'),
        Total_spend=('Dollars', 'sum'),
        Avg_payment_cycle=('payment_cycle', 'mean')
    ).reset_index()
    
    return vendor_summary

## 7. Analyzing Top-Selling Items and Stores

Identify top-performing products and stores to focus on high-revenue areas. The code calculates the total sales for each product and store, then ranks them to identify the top performers.

In [10]:
# Function to analyze top-selling items and stores
def analyze_sales_performance(sales_df):
    """
    Analyze top-selling items and stores.
    
    Args:
        sales_df (pd.DataFrame): The sales DataFrame.
    
    Returns:
        pd.DataFrame: A summary of top-selling items and stores.
    """
    # Top-selling items
    top_items = sales_df.groupby(['Brand', 'Description']).agg(
        Total_sales=('SalesQuantity', 'sum')
    ).reset_index().sort_values(by='Total_sales', ascending=False)
    
    # Top-selling stores
    top_stores = sales_df.groupby('Store').agg(
        Total_sales=('SalesQuantity', 'sum')
    ).reset_index().sort_values(by='Total_sales', ascending=False)
    
    return top_items, top_stores

## 8. Calculating Economic Order Quantity (EOQ)

Determine the optimal order quantity to minimize ordering and carrying costs. The code calculates the EOQ for each product 

In [11]:
# Function to calculate Economic Order Quantity (EOQ)
def calculate_eoq(purchases_df, holding_cost_per_unit, ordering_cost):
    """
    Calculate Economic Order Quantity (EOQ) for each product.
    
    Args:
        purchases_df (pd.DataFrame): The purchases DataFrame.
        holding_cost_per_unit (float): Holding cost per unit per year.
        ordering_cost (float): Cost per order.
    
    Returns:
        pd.DataFrame: A DataFrame with EOQ for each product.
    """
    # Calculate annual demand and order frequency
    demand_summary = purchases_df.groupby(['Brand', 'Description']).agg(
        Annual_demand=('Quantity', 'sum')
    ).reset_index()
    
    # Calculate EOQ
    demand_summary['EOQ'] = np.sqrt(
        (2 * demand_summary['Annual_demand'] * ordering_cost) / holding_cost_per_unit
    ).round(2)
    
    return demand_summary

## 9. Run the Analysis

In [12]:
# Main function to run the analysis
def main():
    # Load data from the zip file
    zip_path = 'Inventory_Analysis_Case_Study.zip'  # Make sure the zip file is in the same directory
    dataframes = load_data_from_zip(zip_path)
    
    if not dataframes:
        logging.error("No data loaded. Exiting.")
        return
    
    # Clean and preprocess data
    for df_name, df in dataframes.items():
        logging.info(f"Cleaning and preprocessing {df_name}...")
        df = clean_data(df)
        
        # Identify date columns dynamically
        date_columns = [col for col in df.columns if 'Date' in col]
        df = preprocess_data(df, date_columns)
        
        # Update the DataFrame in the dictionary
        dataframes[df_name] = df
    
    # Perform inventory analysis
    purchases_df = dataframes['PurchasesFINAL12312016']
    leadtime_summary = analyze_lead_time(purchases_df)
    
    # Calculate safety stock
    sales_df = dataframes['SalesFINAL12312016']
    safety_stock_data = calculate_safety_stock(sales_df, leadtime_summary)
    
    # Identify excess inventory
    inventory_df = dataframes['EndInvFINAL12312016']
    inventory_status = identify_excess_inventory(safety_stock_data, inventory_df)
    logging.info("Excess inventory analysis completed.")
    
    # Analyze vendor performance
    vendor_summary = analyze_vendor_performance(purchases_df)
    logging.info("Vendor performance analysis completed.")
    
    # Analyze top-selling items and stores
    top_items, top_stores = analyze_sales_performance(sales_df)
    logging.info("Top-selling items and stores analysis completed.")
    
    # Calculate EOQ
    holding_cost_per_unit = 5  # Example holding cost per unit per year
    ordering_cost = 50  # Example cost per order
    eoq_summary = calculate_eoq(purchases_df, holding_cost_per_unit, ordering_cost)
    logging.info("EOQ analysis completed.")
    
    # Display results
    print("Top 10 Excess Inventory Items:")
    print(inventory_status[inventory_status['status'] == 'Excess'].head(10))
    
    print("\nTop 10 Vendors by Total Spend:")
    print(vendor_summary.sort_values(by='Total_spend', ascending=False).head(10))
    
    print("\nTop 10 Selling Items:")
    print(top_items.head(10))
    
    print("\nTop 10 Selling Stores:")
    print(top_stores.head(10))
    
    print("\nEOQ Summary:")
    print(eoq_summary.head(10))


In [13]:
# Run the main function
if __name__ == "__main__":
    main()

2025-02-26 13:30:06,213 - INFO - Successfully loaded InvoicePurchases12312016.csv as InvoicePurchases12312016
2025-02-26 13:30:06,506 - INFO - Successfully loaded EndInvFINAL12312016.csv as EndInvFINAL12312016
2025-02-26 13:30:06,744 - INFO - Successfully loaded BegInvFINAL12312016.csv as BegInvFINAL12312016
2025-02-26 13:30:06,761 - INFO - Successfully loaded 2017PurchasePricesDec.csv as 2017PurchasePricesDec
2025-02-26 13:30:07,895 - INFO - Successfully loaded SalesFINAL12312016.csv as SalesFINAL12312016
2025-02-26 13:30:11,540 - INFO - Successfully loaded PurchasesFINAL12312016.csv as PurchasesFINAL12312016
2025-02-26 13:30:11,542 - INFO - Cleaning and preprocessing InvoicePurchases12312016...
2025-02-26 13:30:11,544 - INFO - Handling missing values...
2025-02-26 13:30:11,553 - INFO - Typecasted InvoiceDate to datetime.
2025-02-26 13:30:11,557 - INFO - Typecasted PODate to datetime.
2025-02-26 13:30:11,557 - INFO - Typecasted PayDate to datetime.
2025-02-26 13:30:11,560 - INFO - Cle

Top 10 Excess Inventory Items:
    Brand                   Description  Total_sales  Std_sales  \
0      58   Gekkeikan Black & Gold Sake          288   0.839642   
1      60        Canadian Club 1858 VAP          124   0.521838   
2      61         Margaritaville Silver           24   0.000000   
3      62      Herradura Silver Tequila          162   0.503286   
4      63    Herradura Reposado Tequila          131   0.537436   
5      72          No. 3 London Dry Gin           19   0.543906   
7      77   Three Olives Espresso Vodka          908   1.009252   
8      79      Three Olives Loopy Vodka          416   0.675245   
10    100           Chivas Royal Salute            7   0.000000   
11    104  Mr Boston Wild Cherry Brandy           19   0.543906   

    avg_daily_sales  Max_leadtime  Min_leadtime  Avg_leadtime  Std_leadtime  \
0             4.881            14             3         7.758         2.067   
1             2.102            13             4         7.660         2.6

## Conclusion

**1. Data Quality and Consistency**

- The inventory data was thoroughly cleaned and preprocessed to ensure consistency and reliability. Missing values and duplicates were removed, and date columns were standardized. This ensured that the data was ready for accurate analysis.

**2. Lead Time Analysis**

- The lead time analysis revealed that most suppliers deliver within 7–8 days, with some variability. Products like Gekkeikan Black & Gold Sake and Herradura Silver Tequila had average lead times of 7.76 days and 7.33 days, respectively. However, some products experienced longer lead times, indicating potential inefficiencies in the supply chain.

**3. Safety Stock and Excess Inventory**

- Gekkeikan Black & Gold Sake had an excess inventory of 318 units.
- Three Olives Espresso Vodka had an excess inventory of 1,586 units.

**4. Vendor Performance**

- DIAGEO NORTH AMERICA INC was the top vendor, with total spend of $50,959,796.85.
- The average payment cycle across vendors was 35–37 days.

**5. Top-Selling Items and Stores**

- Smirnoff 80 Proof was the top-selling item, with 28,544 units sold.
- Store 15 was the top-performing store, generating 101,078 units in sales.

**6. Economic Order Quantity (EOQ)**

- Gekkeikan Black & Gold Sake has an EOQ of 266 units.
- Three Olives Espresso Vodka has an EOQ of 468 units.